# Cache Transfer Workflow

This notebook demonstrates a common collaborative workflow using `CacheStack` and `transfer()`:

1. A **shared global cache** holds approved, vetted results (read-only for regular users).
2. Each user runs under a **local cache** stacked on top of the global one.
   - Cache hits in the global cache are served transparently.
   - New computations land only in the local cache.
3. After reviewing their local results, the user selects which ones to **promote** to the global cache via `transfer()`.

## 1. Setup

We create two caches:

- `_global_cache_rw` — a persistent SQL + PickleFile cache simulating a shared remote store (admin access only)
- `global_cache` — a read-only wrapper that regular users see (writes rejected), created via `.readonly()`
- `local_cache` — a fast in-memory cache for the user's current session
- `user_cache` — local stacked on top of global via `.push()`, so local results take priority

In [ ]:
import tempfile
import os
import time

from fleche import fleche, cache
from fleche.caches import Cache
from fleche.storage import Memory
from fleche.storage.pickle_file import PickleFile
from fleche.storage.sql import Sql

The admin-accessible cache uses persistent SQL + PickleFile storage.

In [ ]:
tmp_dir = tempfile.TemporaryDirectory()

_global_cache_rw = Cache(
    values=PickleFile.with_cloudpickle(tmp_dir.name),
    _calls=Sql(f"sqlite:///{os.path.join(tmp_dir.name, 'global.db')}"),
)
_global_cache_rw

Regular users get a read-only view via `.readonly()`, and a per-session in-memory cache stacked on top via `.push()`.

In [ ]:
global_cache = _global_cache_rw.readonly()
local_cache = Cache(values=Memory({}), _calls=Memory({}))
user_cache = global_cache.push(local_cache)
user_cache

## 2. Define Functions

Two `@fleche`-decorated functions simulate a heavy computation and a post-processing step.

In [ ]:
@fleche
def simulate(param: float) -> dict:
    """Expensive simulation — takes time, results worth sharing."""
    print(f"  [simulate] Running simulation for param={param}...")
    time.sleep(0.05)  # pretend this is expensive
    return {"param": param, "result": param ** 2 + 1.0}


@fleche
def postprocess(data: dict) -> float:
    """Quick post-processing — user-specific, not worth sharing."""
    print(f"  [postprocess] Processing {data}...")
    return data["result"] * 2.0

## 3. Seed the Global Cache (Admin Step)

An admin pre-populates the global cache with approved baseline results.
Regular users never do this — they only read from `global_cache` (the read-only view).

In [ ]:
with cache(_global_cache_rw):
    for p in [1.0, 2.0, 3.0]:
        simulate(p)

In [ ]:
_global_cache_rw.table()

## 4. User Session

The user runs under `user_cache`, a `CacheStack` with local priority over global.

- Params **already in the global cache** (1.0, 2.0, 3.0) → cache hits, no recomputation.
- **New params** (4.0, 5.0, 6.0) → computed and saved to `local_cache` only.
- Post-processing results also land in `local_cache`.

Params 1–3 are already in the global cache — no `[simulate]` output means they are cache hits.

In [ ]:
with cache(user_cache):
    for p in [1.0, 2.0, 3.0]:
        postprocess(simulate(p))

Params 4–6 are new — they will be computed and stored in `local_cache` only.

In [ ]:
with cache(user_cache):
    for p in [4.0, 5.0, 6.0]:
        postprocess(simulate(p))

## 5. Inspect the Caches

After the session:
- The **global cache** is unchanged (still only has params 1–3).
- The **local cache** has all new results (simulate + postprocess for params 4–6, and postprocess for 1–3 which were hits from global).

In [ ]:
_global_cache_rw.table()

In [ ]:
local_cache.table()

## 6. Filter Before Transfer

The user reviews their local cache and decides only the `simulate` results are worth promoting to the global cache.
Post-processing results are user-specific and stay local.

`filter()` returns a `FilteredCache` — a read-only view that `transfer()` will iterate over.

In [ ]:
simulations_only = local_cache.filter(lambda c: c.name == "simulate")
simulations_only.table()

## 7. Transfer to the Global Cache

The user (or admin) calls `transfer()` to promote the selected results into the global cache.
With the default `overwrite=False`, entries already present in the global cache are left untouched.
After this, any user will get cache hits for `simulate(4.0)`, `simulate(5.0)`, and `simulate(6.0)` without recomputation.

In [ ]:
simulations_only.transfer(_global_cache_rw)

In [ ]:
_global_cache_rw.table()

## 8. Verify: Second User Gets Cache Hits

A second user starts a fresh session with their own empty local cache.
They can now load `simulate(4.0)` – `simulate(6.0)` without any computation.

In [ ]:
local_cache_2 = Cache(values=Memory({}), _calls=Memory({}))
user_cache_2 = global_cache.push(local_cache_2)

All six params should now be cache hits — no `[simulate]` output expected.

In [ ]:
with cache(user_cache_2):
    for p in [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]:
        simulate(p)

In [ ]:
tmp_dir.cleanup()